In [1]:
import os
import json
import pandas as pd
import traceback

In [19]:
from langchain_openai import ChatOpenAI

In [58]:
from dotenv import load_dotenv
load_dotenv()

True

In [65]:
KEY=os.getenv("OPENAI_API_KEY")

In [66]:
llm=ChatOpenAI(openai_api_key=KEY,model_name="gpt-3.5-turbo",temperature=0.5)

In [9]:
llm

ChatOpenAI(output_version=None, profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'attachment': False, 'temperature': True, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002E7D3F91610>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002E7D48FC250>, root_client=<openai.OpenAI object at 0x000002E7D3F91150>, root_async_client=<openai.AsyncOpenAI object at 0x000002E7D48EBD90>, temperature=0.5, model_kwargs={}, o

In [25]:
from langchain_openai import OpenAI

In [26]:
import PyPDF2

In [27]:
from langchain_core.prompts import PromptTemplate

In [29]:
RESPONSE_JSON={
    "1":{
        "mcq":"multiple choice question ",
        "options":{
            "a":"choice here",
            "b":"choice here",
            "c":"choice here",  
            "d":"choice here",
        },
        "correct":"correct answer",
    },
    "2":{
        "mcq":"multiple choice question ",
        "options":{
            "a":"choice here",
            "b":"choice here",
            "c":"choice here",  
            "d":"choice here",
        },
        "correct":"correct answer",
    },
    "3":{
        "mcq":"multiple choice question ",
        "options":{
            "a":"choice here",
            "b":"choice here",
            "c":"choice here",  
            "d":"choice here",
        },
        "correct":"correct answer",
    }
}

In [34]:
TEMPLATE="""
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to create a quiz of {number} multiple choice questions for the subject of {subject} in {tone} tone. Make sure the questions are not repeated and check all the questions to be conforming the text as well. Make sure to format your response like RESPONSE_JSON below and use it as a guide.\
Ensure to make {number} MCQs
###RESPONSE_JSON
{response_json}
"""

In [35]:
quiz_generation_prompt=PromptTemplate(
    input_variables=["text","number","subject","tone","response_json"],
    template=TEMPLATE
)

In [42]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
import langchain

In [67]:
modern_chain=prompt=quiz_generation_prompt |llm| RunnableParallel(quiz=StrOutputParser())

In [40]:
TEMPLATE2="""
You are an expert english grammarian and writer.given a mcq for {subject} subject\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity.If the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which need to be changed and change the tone such that it perfectly fits the student abilities.
Quiz_MCQs:
{quiz}
Check from an expert English writer of the above quiz:
"""

In [41]:
quiz_evaluation_prompt=PromptTemplate(
    input_variables=["subject","quiz"],
    template=TEMPLATE2
)

In [68]:
quiz_evaluation_chain=prompt=quiz_evaluation_prompt|llm|RunnableParallel(review=StrOutputParser())

In [69]:
sequential_chain=modern_chain|quiz_evaluation_chain

In [49]:
file_path=r"C:\Users\shrey\OneDrive\Desktop\mcqgen\data.txt"

In [52]:
with open(file_path,"r") as file:
    TEXT=file.read()

In [53]:
#serialize python dictionary to json string
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question ", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question ", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question ", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [55]:
from langchain_community.callbacks import get_openai_callback

In [56]:
NUMBER=5
SUBJECT="machine learning"
TONE="simple"

In [70]:
with get_openai_callback() as cb:
    response = sequential_chain.invoke(
        {
            "text": TEXT,
            "number":NUMBER,
            "subject":SUBJECT,
            "tone":TONE,
            "response_json":json.dumps(RESPONSE_JSON),
        }
    )
    
    print("\n--- Final Chain Output ---")
    print(response)
    print("\n--- OpenAI Metrics ---")
    print(f"Total Tokens: {cb.total_tokens}")
    print(f"Prompt Tokens: {cb.prompt_tokens}")
    print(f"Completion Tokens: {cb.completion_tokens}")
    print(f"Total Cost (USD): ${cb.total_cost}")

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
print("\n--- OpenAI Metrics ---")
print(f"Total Tokens: {cb.total_tokens}")
print(f"Prompt Tokens: {cb.prompt_tokens}")
print(f"Completion Tokens: {cb.completion_tokens}")
print(f"Total Cost (USD): ${cb.total_cost}")

In [ ]:
quiz=response.get("quiz")

In [ ]:
quiz_table_data=[]
for key, value in quiz.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}:{option_value}"
            for option, option_value in value["options"].items()]
        )
    correct = value["correct"]
    
    quiz_table_data.append({"MCQ":mcq,"Choices": options, "Correct Answer": correct})

In [ ]:
quiz=pd.DataFrame(quiz_table_data)

In [ ]:
quiz.to_csv("machinelearning.csv", index=False)